<a href="https://colab.research.google.com/github/pedromoreira427/agroguard-fiap/blob/main/pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

print("✅ Bibliotecas carregadas!")

✅ Bibliotecas carregadas!


In [2]:
from google.colab import drive
drive.mount('/content/drive')

df_focos = pd.read_csv("/content/drive/MyDrive/agroguard/focos_cerrado_2020_2024.csv")
df_clima = pd.read_csv("/content/drive/MyDrive/agroguard/clima_cerrado_2020_2024.csv")

print(f"Focos: {len(df_focos)} registros")
print(f"Clima: {len(df_clima)} registros")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Focos: 273308 registros
Clima: 1827 registros


In [3]:
# Garante que a coluna de data está no formato correto
df_focos['acq_date'] = pd.to_datetime(df_focos['acq_date'])

# Agrupa por data e calcula métricas diárias
df_focos_dia = df_focos.groupby('acq_date').agg(
    quantidade_focos  = ('frp', 'count'),   # total de focos no dia
    intensidade_media = ('brightness', 'mean'),  # temperatura média dos focos
    frp_medio         = ('frp', 'mean'),    # potência média do fogo
    frp_total         = ('frp', 'sum'),     # potência total do fogo no dia
    confianca_media   = ('confidence', 'mean')   # confiança média da detecção
).reset_index()

df_focos_dia.columns = ['data', 'quantidade_focos', 'intensidade_media',
                         'frp_medio', 'frp_total', 'confianca_media']

print(f"✅ {len(df_focos_dia)} dias com focos registrados")
print(df_focos_dia.head())

✅ 1763 dias com focos registrados
        data  quantidade_focos  intensidade_media  frp_medio  frp_total  \
0 2020-01-01                26         315.988462  17.303846      449.9   
1 2020-01-03                 8         317.075000  13.962500      111.7   
2 2020-01-05                 1         320.200000  10.700000       10.7   
3 2020-01-06                 1         317.300000   6.800000        6.8   
4 2020-01-07                 2         314.750000  17.950000       35.9   

   confianca_media  
0        48.923077  
1        55.375000  
2        46.000000  
3        43.000000  
4        51.000000  


In [4]:
df_clima['data'] = pd.to_datetime(df_clima['data'])

print(f"✅ Dados climáticos: {len(df_clima)} dias")
print(df_clima.head())

✅ Dados climáticos: 1827 dias
        data  latitude  longitude  temp_max  precipitacao  vento_max  \
0 2020-01-01    -15.78     -47.93     25.95     20.800001   9.504272   
1 2020-01-02    -15.78     -47.93     26.00     20.100000  15.716793   
2 2020-01-03    -15.78     -47.93     25.35     16.100002  17.786331   
3 2020-01-04    -15.78     -47.93     24.05      8.599999  22.183128   
4 2020-01-05    -15.78     -47.93     23.60     21.500002  21.129883   

   umidade_max  
0    95.409310  
1    95.434930  
2    96.920860  
3    95.133575  
4    95.725520  


In [5]:
# Cria um calendário com todos os dias de 2020 a 2024
todos_os_dias = pd.DataFrame({
    'data': pd.date_range(start='2020-01-01', end='2024-12-31', freq='D')
})

# Cruza com os focos (dias sem foco ficam com 0)
df = todos_os_dias.merge(df_focos_dia, on='data', how='left')
df['quantidade_focos'] = df['quantidade_focos'].fillna(0)
df['intensidade_media'] = df['intensidade_media'].fillna(0)
df['frp_medio']         = df['frp_medio'].fillna(0)
df['frp_total']         = df['frp_total'].fillna(0)
df['confianca_media']   = df['confianca_media'].fillna(0)

# Cruza com o clima
df = df.merge(df_clima, on='data', how='left')

print(f"✅ Dataset combinado: {len(df)} dias")
print(df.head())

✅ Dataset combinado: 1827 dias
        data  quantidade_focos  intensidade_media  frp_medio  frp_total  \
0 2020-01-01              26.0         315.988462  17.303846      449.9   
1 2020-01-02               0.0           0.000000   0.000000        0.0   
2 2020-01-03               8.0         317.075000  13.962500      111.7   
3 2020-01-04               0.0           0.000000   0.000000        0.0   
4 2020-01-05               1.0         320.200000  10.700000       10.7   

   confianca_media  latitude  longitude  temp_max  precipitacao  vento_max  \
0        48.923077    -15.78     -47.93     25.95     20.800001   9.504272   
1         0.000000    -15.78     -47.93     26.00     20.100000  15.716793   
2        55.375000    -15.78     -47.93     25.35     16.100002  17.786331   
3         0.000000    -15.78     -47.93     24.05      8.599999  22.183128   
4        46.000000    -15.78     -47.93     23.60     21.500002  21.129883   

   umidade_max  
0    95.409310  
1    95.434930 

In [6]:
# Mês e dia do ano (sazonalidade)
df['mes']        = df['data'].dt.month
df['dia_do_ano'] = df['data'].dt.dayofyear

# Dias consecutivos sem chuva
df['dias_sem_chuva'] = 0
contador = 0
for i, row in df.iterrows():
    if row['precipitacao'] == 0 or pd.isna(row['precipitacao']):
        contador += 1
    else:
        contador = 0
    df.at[i, 'dias_sem_chuva'] = contador

# Média de focos nos últimos 7 dias (tendência recente)
df['media_focos_7d'] = (
    df['quantidade_focos']
    .rolling(window=7, min_periods=1)
    .mean()
    .round(2)
)

print("✅ Features extras criadas!")
print(df[['data', 'mes', 'dias_sem_chuva', 'media_focos_7d']].head(10))

✅ Features extras criadas!
        data  mes  dias_sem_chuva  media_focos_7d
0 2020-01-01    1               0           26.00
1 2020-01-02    1               0           13.00
2 2020-01-03    1               0           11.33
3 2020-01-04    1               0            8.50
4 2020-01-05    1               0            7.00
5 2020-01-06    1               0            6.00
6 2020-01-07    1               0            5.43
7 2020-01-08    1               0            2.86
8 2020-01-09    1               0            3.29
9 2020-01-10    1               0           10.43


In [7]:
def classificar_risco(focos):
    if focos <= 50:
        return 0  # baixo
    elif focos <= 200:
        return 1  # médio
    else:
        return 2  # alto

df['risco'] = df['quantidade_focos'].apply(classificar_risco)

# Mostra a distribuição dos níveis de risco
print("📊 Distribuição dos níveis de risco:")
contagem = df['risco'].value_counts().sort_index()
for nivel, qtd in contagem.items():
    nome = ['Baixo', 'Médio', 'Alto'][nivel]
    pct = qtd / len(df) * 100
    print(f"  {nivel} - {nome}: {qtd} dias ({pct:.1f}%)")

📊 Distribuição dos níveis de risco:
  0 - Baixo: 823 dias (45.0%)
  1 - Médio: 625 dias (34.2%)
  2 - Alto: 379 dias (20.7%)


In [8]:
# Remove colunas derivadas dos focos — o modelo não pode usar isso para prever
df_modelo = df.drop(columns=[
    'data',
    'latitude',
    'longitude',
    'quantidade_focos',    # base do cálculo do risco
    'intensidade_media',   # só existe depois do foco
    'frp_medio',           # só existe depois do foco
    'frp_total',           # só existe depois do foco
    'confianca_media',     # só existe depois do foco
    'media_focos_7d'       # derivado dos focos passados
], errors='ignore')

# Remove linhas com valores nulos
df_modelo = df_modelo.dropna()

df_modelo.to_csv("/content/drive/MyDrive/agroguard/dataset_agroguard.csv", index=False)

print(f"✅ Dataset corrigido: {len(df_modelo)} registros")
print(f"Colunas finais: {list(df_modelo.columns)}")

✅ Dataset corrigido: 1827 registros
Colunas finais: ['temp_max', 'precipitacao', 'vento_max', 'umidade_max', 'mes', 'dia_do_ano', 'dias_sem_chuva', 'risco']


In [9]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs("/content/drive/MyDrive/agroguard", exist_ok=True)

df_modelo.to_csv("/content/drive/MyDrive/agroguard/dataset_agroguard.csv", index=False)

print("💾 dataset_agroguard.csv salvo no Drive!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
💾 dataset_agroguard.csv salvo no Drive!
